# Day I12 — gait event 검출: 문턱 하나에도 규칙이 필요하다

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I12.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I12.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 검출 함수를 모듈로 만들기 (`gait_events.py`)

아래 코드를 `notebooks/advanced/I/gait_events.py`로 저장합니다. Day I13·I14에서 `from gait_events import strides`로 다시 씁니다.

In [ ]:
import numpy as np

def contact_mask(force, thr, fs, min_swing=0.1, min_contact=0.2):
    """force > thr 이면 접지(True). 너무 짧은 유각(깜박임)은 접지로 메우고,
    너무 짧은 접지(튀는 값)는 유각으로 돌린다."""
    on = force > thr
    for value, min_len in [(False, min_swing), (True, min_contact)]:
        edges = np.flatnonzero(np.diff(on.astype(int))) + 1
        starts = np.r_[0, edges]
        ends = np.r_[edges, len(on)]
        for s, e in zip(starts, ends):
            if on[s] == value and (e - s) < min_len * fs and s > 0 and e < len(on):
                on[s:e] = not value
    return on

def strides(force, thr, fs, drop_ends=2):
    """stride 표: 열 = [IC 시각(s), stride time, stance time, swing time]"""
    on = contact_mask(force, thr, fs)
    ic = np.flatnonzero(~on[:-1] & on[1:]) + 1      # 공중 → 접지
    to = np.flatnonzero(on[:-1] & ~on[1:]) + 1      # 접지 → 공중
    rows = []
    for a, b in zip(ic[:-1], ic[1:]):
        t_off = to[(to > a) & (to < b)]
        if len(t_off) == 1:                         # stride 안에 TO가 정확히 하나일 때만
            rows.append((a / fs, (b - a) / fs, (t_off[0] - a) / fs, (b - t_off[0]) / fs))
    rows = np.array(rows)[drop_ends:len(rows) - drop_ends]   # 규칙 1: 앞뒤 stride 제외
    med = np.median(rows[:, 1])
    mad = 1.4826 * np.median(np.abs(rows[:, 1] - med))
    ok = np.abs(rows[:, 1] - med) <= 5 * mad                  # 규칙 2: 이상 stride 제외
    return rows, ok

### 예시 2 — 실제 기록에 적용하고 문턱 민감도 보기

In [ ]:
import numpy as np
from gait_events import contact_mask, strides

fs = 100
d = np.loadtxt("data/gaitpdb/GaCo01_01.txt")
for col, name in [(17, "왼발"), (18, "오른발")]:
    raw = d[:, col] > 20
    n_raw = np.sum(~raw[:-1] & raw[1:])
    on = contact_mask(d[:, col], 20, fs)
    n_clean = np.sum(~on[:-1] & on[1:])
    print(f"{name}: 20 N 문턱 그대로 IC {n_raw}개 → 깜박임 정리 후 {n_clean}개")

left = d[:, 17]
rows, ok = strides(left, 20, fs)
print(f"\n왼발 stride {len(rows)}개(앞뒤 2개씩 제외 후) → 이상 stride 제외 후 {ok.sum()}개")
for r in rows[~ok]:
    print(f"  제외: {r[0]:6.2f} s 시작, stride {r[1]:.2f} s")
r = rows[ok]
print(f"평균 stride {r[:, 1].mean():.3f} s | stance {r[:, 2].mean():.3f} s ({100 * np.mean(r[:, 2] / r[:, 1]):.1f}%) | swing {r[:, 3].mean():.3f} s")

p95 = np.percentile(left, 95)          # 그 발 힘의 95번째 백분위수 ≈ 봉우리 높이
print(f"\n민감도 (기준: 왼발 힘 95번째 백분위수 {p95:.0f} N)")
print("문턱            stride  stride(s)  stance(%)  swing(s)")
for frac in [0.01, 0.025, 0.05, 0.10, 0.20]:
    rows, ok = strides(left, frac * p95, fs)
    r = rows[ok]
    print(f"{100 * frac:4.1f}% ({frac * p95:5.1f} N)   {ok.sum():3d}    {r[:, 1].mean():.3f}     {100 * np.mean(r[:, 2] / r[:, 1]):5.1f}     {r[:, 3].mean():.3f}")